# Train a Bounded CT Denoiser

Dataset : CT (2D slices sampled from volumes)

Pipelines:
 - CT preprocessing
 -  Train resiudal-form denoiser

Outputs:
 - checkpoint (.pt)
 - logs.csv

### train_bounded_denoiser_ct

In [ ]:
!pip -q install kaggle opencv-python matplotlib pandas

In [ ]:
!pip install nibabel pydicom

In [ ]:
!mkdir -p models
!mkdir -p outputs/checkpoints
!mkdir -p outputs/logs

In [ ]:
import json

with open("kaggle.json", "r") as f:
    data = json.load(f)

print("keys:", list(data.keys()))

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p ~/data/raw ~/data/extracted
!kaggle datasets download -d zofiaknapiska/spider-lumbar-spine-segmentation-in-mr-nifti -p ~/data/raw
!unzip -q ~/data/raw/spider-lumbar-spine-segmentation-in-mr-nifti.zip -d ~/data/extracted/spider

In [ ]:
import os, sys, glob, csv, math
import random


import torch
import torch.nn as nn
import pandas as pd



import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.parametrizations import spectral_norm

from safetensors.torch import save_file, load_file
import matplotlib.pyplot as plt
from tqdm import tqdm

# jax
from jax import jvp, grad
import jax.numpy as jnp

print('torch:', torch.__version__)

In [ ]:
# import models

from src.models.unet2d import UNet2d
from src.models.vit import ViTEndPointDenoiser
from src.models.diffusion import EMPPre

In [ ]:
import glob
import nibabel as nib
import os

base = '/content/data/extracted/spider'

# Find NIfTI files, including both .nii and .nii.gz extensions
nii_files = glob.glob(os.path.join(base, '**', '*.nii'), recursive=True)
nii_files.extend(glob.glob(os.path.join(base, '**', '*.nii.gz'), recursive=True))

print(f'Found {len(nii_files)} NIfTI files. First 10: {nii_files[:10]}')

if nii_files:
    # Load the first NIfTI image found
    image = nib.load(nii_files[0])
    print(f'Loaded: {nii_files[0]}')
    print(f'Image shape: {image.shape}')
    print(f'Image dtype: {image.get_fdata().dtype}')
else:
    print("No NIfTI files were found in the specified directory. Please check the path and file extensions.")

In [ ]:
def seed_all(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

def get_device(device='cuda'):
  if torch.cuda.is_available():
    return torch.device(device)

  if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    return torch.device('mps')

  return torch.device('cpu')

In [ ]:
trajectory_base = '/content/data/extracted/trajectory'

cfg = {
    'seed':42,
    'exp_id' : 'pnp_endpoint_compare',
    'out_dir' : '../outputs/results',
    'ckpt_safetensors' : '../outputs/checkpoints/vit_endpoint/model.safetensors',
    'ckpt_dir': '../outputs/checkpoints/steps.pt',
    'out_npz' : '../outputs/trajactory/pnp_tr.npz',
    'device': 'cuda',
    'data': {
        'root': trajectory_base,
        'format': 'nifti',
        'img_size' : 256,
        'channels' : 1,
        'batch_size' : 16,
        'shuffle' : True,
        'drop_last' : True,
        'num_workers' : 2,
        'hu_min': -1000,
        'hu_max': 400,
        'min_nonair_frac' : .10
    },
    'noise': {
        'sigma_min': .01,
        'sigma_max' : .2,
        'dist' : 'loguniform',
    },
    'pnp' : {
      'iters' : 50,
      'rho0' : .5,
      'gamma' : 1.01,
      'lam' : 1.0,
      'total_primal' : 1e-5,
      'total_fp': 1e-5,
      'save_every' : 1,
    },
    'fixed_t': {
        't_fix': .1,
    },
    'adaptive_rho': {
        't_min': .02,
        't_max': .3,
        'alpha': 1.0,
    },
    'model':{
        'name': 'ViT2D',
        'spectral_norm' : {'enabled': True, 'c': .95},
    },
    'loss':{
        'base' : 'eps_mse',
        'jacobian' : {'enabled': True, 'beta': 1e-3, 'n_samples': 1},
    },
    'vit': {
        'dim': 768,
        'depth': 12,
        'heads': 12,
        'mlp_dim': 3072,
        'mlp_ratio': 3.0,
        'dropout': 0.1,
        'patch': 16,
        'channels': 1,
    },
    'operator': {
        'type': 'inpaint',
        'noise_sigma': .01,
        'mask_ratio' : .5,
    },
    'train':{
        "steps": 20000,
        "lr": 1e-4,
        "weight_decay": 0.0,
        "amp": True,
        "ema": True,
        "ema_decay": 0.999,
        "log_every": 100,
        "save_every": 2000,
        "jac_beta":1e-3,
        "jac_samples":1,
        "out_dir": "../outputs/checkpoints",
        "log_dir": "../outputs/logs",
    },
}

seed_all(cfg["seed"])
device = get_device(cfg["device"])

device

In [ ]:
# utils
def hu_win_normalize(img_hu: np.ndarray, hu_min:float, hu_max: float) ->  np.ndarray:
  x = np.clip(img_hu, hu_min, hu_max)
  x = (x - hu_min) / (hu_max - hu_min + 1e-12)
  x = x * 2.0 - 1.0
  return x.astype(np.float32)

def nonair_fraction(x: np.ndarray, threshold : float = -.95) ->  float:
  return float((x > threshold).mean())

In [ ]:
class CTSliceDatasetNifti(Dataset):
  def __init__(self, root, img_size, channels, hu_min, hu_max, min_nonair_frac=.1, max_steps=20):
    import nibabel as nib
    self.nib = nib
    self.paths = sorted(glob.glob(os.path.join(root, '**', '*.nii*'), recursive=True))
    assert len(self.paths) > 0 , f'No Nifti found under {root}'

    self.img_size = img_size
    self.hu_min = hu_min
    self.hu_max = hu_max
    self.min_nonair_frac = min_nonair_frac
    self.max_steps = max_steps

  def __len__(self):
    return len(self.paths)

  def _load_vol(self, path):
    vol = self.nib.load(path).get_fdata()
    if vol.ndim == 4:
      vol = vol[..., 0]

    vol = np.transpose(vol, (2, 0, 1))
    print(f'vol shape: {vol.shape} must be [B, L, W]')
    return vol

  def _resize(self, x):
    x_t = torch.from_numpy(x)[None, None]
    x_t = torch.nn.functional.interpolate(x_t, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
    return x_t[0, 0].numpy()

  def __getitem__(self, idx):
    path = self.paths[idx]
    vol = self._load_vol(path) # (D, L, W)

    for _ in range(self.max_steps):
      z = np.random.randint(0, vol.shape[0])
      sl = vol[z]
      sl = self._resize(sl)
      sl = hu_win_normalize(sl, self.hu_min, self.hu_max)
      print(f'sl shape: {sl.shape}')
      if nonair_fraction(sl) >= self.min_nonair_frac:
        x = torch.from_numpy(sl)[None, ...] # [1, L, W]
        return x

    x = torch.from_numpy(sl)[None, ...]

def build_dataloader(cfg):
    format = cfg['data']['format']
    if format == 'nifti':
      dataset = CTSliceDatasetNifti(
          root=cfg['data']['root'],
          img_size=cfg['data']['img_size'],
          hu_min=cfg['data']['hu_min'],
          hu_max=cfg['data']['hu_max'],
          min_nonair_frac=cfg['data']['min_nonair_frac'],
          channels=cfg['data']['channels'],
      )
    else:
      raise NotImplementedError(f'format {format} not implemented')

    return DataLoader(dataset, batch_size=cfg['data']['batch_size'], shuffle=cfg['data']['shuffle'], drop_last=cfg['data']['drop_last'], num_workers=0)

loader = build_dataloader(cfg)
batch = next(iter(loader))
print(f'batch shape: {batch.shape}')

In [ ]:
# expected sigma: [b, 1, 1, 1]

def sample_sigma(batch, sigma_min, sigma_max, device):
    u = torch.randn(batch, device=device)
    uu = sigma_min * (sigma_max / sigma_min) ** u

    return uu

def to_sigma_data(batch, sigma):
    sigma_data = sigma.view(batch, 1, 1, 1)

    return sigma_data

In [ ]:
def build_denoiser(kind: str, cfg) -> nn.Module:

  if kind == 'unet':
    from models.unet import UNet
    return UNet(in_channels=cfg['data']['channels'], out_channels=cfg['data']['channels'])

  if kind == 'vit2d':
    from models.vit2d import ViT2D # ViTEndPointDenoiser
    return ViT2D(in_channels=cfg['data']['channels'], out_channels=cfg['data']['channels'], img_size=cfg['data']['img_size'])

  print(f'model kind: {kind}')

def apply_spectral_norm(model, c=1.0):
  for m in model.modules():
    if isinstance(m, nn.Conv2d):
      spectral_norm(m)
  if c < 1.0:
    for p in model.parameters():
      p.data.mul_(c)

  return model


In [ ]:
def sample_xt_sde(x_0, t_min:float=1e-3, t_max:float=1.0):
  """
  score based diffusion : variance preserving sde
  """
  batch_size = x_0.shape[0]
  device = x_0.device
  t = torch.rand(batch_size, device=device) * (t_max - t_min) + t_min

  # beta(t) = beta_0 + t * (bata_1 - bata_0)
  beta_0 = 0.1
  beta_1 = 20.0

  # log_mean_coeff = -1/4 * t^2 * (beta_1 - beta_0) - 1/2 * t * beta_0
  # exp(log_mean_coeff) = sqrt(alpha_bar_t)
  log_mean_coeff = -0.25 * t ** 2 * (beta_1 - beta_0) - .6 * t * beta_0
  alpha_t = torch.exp(log_mean_coeff).view(-1, 1, 1, 1)
  std_t = torch.sqrt(1 - torch.exp(2.0 * log_mean_coeff)).view(-1, 1, 1, 1)

  eps = torch.randn_like(x_0)
  x_t = alpha_t * x_0 + std_t * eps

  return x_t, t, x_0

In [ ]:
def jacobian_penalty_residual(model, x_t, t, n_samples=1):
  """
  L_jacobian = E ||J_R(x_t, t) z||^2, R = x_t - D(x_t, t)
  Using PyTorch autograd for compatibility.
  """
  reg = 0.0
  x_t = x_t.detach().requires_grad_(True)

  for _ in range(n_samples):
    z = torch.randn_like(x_t)

    # D(x_t, t)
    d_out = model(x_t, t)
    # R(x_t) = x_t - D(x_t, t)
    residual = x_t - d_out

    # Compute Jacobian-Vector Product (JVP): (dR/dx) * z
    # We use torch.autograd.grad to get the gradient of (residual * z) w.r.t x_t
    vjp = torch.autograd.grad(
        outputs=residual,
        inputs=x_t,
        grad_outputs=z,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    reg = reg + vjp.pow(2).mean()

  return reg / n_samples

def jacobian_loss(model, x_noisy, sigma, key):
    def get_sn(x_s, s_s):
        v = jax.random.normal(key, x_s.shape)
        _, jvp_val = jax.jvp(lambda z: model(z, s_s), (x_s,), (v,))
        return jnp.linalg.norm(jvp_val)**2

    return jnp.mean(jax.vmap(get_sn)(x_noisy, sigma))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure directory exists
os.makedirs("/content/outputs/logs", exist_ok=True)
os.makedirs("/content/outputs/checkpoints", exist_ok=True)

x_0 = next(iter(loader))
x_0 = x_0.to(device)
B = x_0.shape[0]

backbone = build_denoiser(cfg['model']['name'].lower(), cfg).to(device)

model = EMPPre(backbone, sigma_data=cfg['data']['sigma_data'])

sigma = sample_sigma(cfg['data']['batch_size'], cfg['noise']['sigma_min'], cfg['noise']['sigma_max'])
sigma = to_sigma_data(sigma)

eps = torch.randn_like(x_0)
x_noisy = x_0 + sigma * eps


optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
# Use the newer torch.amp API
scaler = torch.amp.GradScaler('cuda', enabled=cfg['train']['amp'] and device.type=='cuda')

if cfg['model']['spectral_norm']['enabled']:
  model = apply_spectral_norm(model, cfg['model']['spectral_norm']['c'])

log_path = os.path.join("/content/outputs/logs", f'{cfg["exp_id"]}.csv')
with open(log_path, 'w', newline='') as f:
  csv.writer(f).writerow(['step', 'loss', 'rec', 'jac'])

data_iter = iter(loader)
model.train()

pbar = tqdm(range(1, cfg['train']['steps'] + 1), desc=f'training: {cfg["exp_id"]}')

for step in pbar:
  try:
    batch = next(data_iter)
  except StopIteration:
    data_iter = iter(loader)
    batch = next(data_iter)

  x_0 = batch.to(device)
  x_t, t, x0_target = sample_xt_sde(x_0)

  optimizer.zero_grad(set_to_none=True)

  with torch.amp.autocast('cuda', enabled=cfg['train']['amp']):
    x0_hat = model(x_t, t)
    rec_loss = F.mse_loss(x0_hat, x0_target)

    # Now using the fixed PyTorch-based penalty
    jac_reg = jacobian_penalty_residual(
        model, x_t, t, n_samples=cfg['train']['jac_samples']
    )

    total_loss = rec_loss + cfg['train']['jac_beta'] * jac_reg

  scaler.scale(total_loss).backward()
  scaler.step(optimizer)
  scaler.update()

  if step % cfg['train']['log_every'] == 0:
    metrics = {'loss': total_loss.item(), 'rec': rec_loss.item(), 'jac': jac_reg.item()}
    pbar.set_postfix({k: f'{v:.2e}' for k, v in metrics.items()})
    with open(log_path, 'a', newline='') as f:
      csv.writer(f).writerow([step, *metrics.values()])

  if step % cfg['train']['save_every'] == 0:
    ckpt_path = os.path.join("/content/outputs/checkpoints", f'{cfg["exp_id"]}_{step}.pt')
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'cfg' : cfg
    }, ckpt_path)
    print(f'Saved: {ckpt_path}')

In [ ]:
def t_mapping_linear(sigma, eps=1e-12, filp=False):
  sigma_t = np.asarray(sigma, dtype=np.float32)
  t = (sigma_t - sigma_t.min()) / (sigma_t.max() - sigma_t.min() + eps)
  t = np.clip(t, 0.0, 1.0)
  if filp:
    t = 1.0 - t

  return t

def t_mapping_log(sigma, eps=1e-12, filp=False):
  sigma_t = np.asarray(sigma, dtype=np.float64)
  sigma_t = np.maximum(sigma_t, eps)
  ls = np.log(sigma_t)
  t = (ls - ls.min()) / (ls.max() - ls.min() + eps)
  t = np.clip(t, 0.0, 1.0)

  if filp:
    t = 1.0 - t

  return t

def t_mapping_power(sigma, alpha=1.0, use_log=True, flip=False):
  if use_log:
    t = t_mapping_log(sigma, filp=True)
  else:
    t = t_mapping_linear(sigma, filp=False)

  t = np.clip(t, 0.0, 1.0) ** alpha

  return 1.0 - t if flip else t

In [ ]:
def t_mapping_log_continuous(sigma:torch.Tensor, sigma_min:float, sigma_max:float, eps:float = 1e-12) -> torch.Tensor:
  sigma_0 = sigma.clamp_min(eps)
  t = (
      sigma_0.log() - torch.log(torch.tensor(sigma_min, device = sigma_0.device))
  ) / (torch.log(torch.tensor(sigma_max, device=sigma_0.device)) - torch.log(torch.tensor(sigma_min, device=sigma_0.device))+eps)

  return t.clamp(0.0, 1.0)

def t_mapping_log_idx(sigma:torch.Tensor, sigma_min:float, sigma_max:float, T: int, eps:float = 1e-12) -> torch.Tensor:
  t = t_mapping_log_continuous(sigma, sigma_min, sigma_max, eps=eps)
  idx = torch.round(t * (T - 1)).long()
  return idx.clamp(0, T -1)

In [ ]:
def t_fixed(_k, _rho, _primal, cfg):
  return float(cfg['fixed_t']['t_fix'])

def t_rho_based(_k, rho, _primal, cfg):
  t_min, t_max = cfg['adaptive_rho']['t_min'], cfg['adaptive_rho']['t_max']
  alpha = cfg['adaptive_rho']['alpha']
  rho0 = cfg['pnp']['rho0']
  value = t_min + (t_max - t_min) * (rho0/rho) ** alpha

  return float(max(t_min, min(t_max, value)))

In [ ]:
# unet denoiser
unet = UNet2d(in_channels=1, out_channels=1, base_channels=16, num_levels=4)

denoiser_1 = unet().to(device).eval()
ckpt = torch.load(cfg['ckpt_dir'], map_location='cpu')
denoiser_1.load_state_dict(ckpt['model'], strict=True)
denoiser_1.eval()


vit = ViTEndpointDenoiser(img_size=cfg['data']['img_size'], patch=16, in_channel=cfg['data']['in_channel'],
                            dim=512, depth=8, heads=8, mlp_ratio=cfg['vit']['mlp_ratio'], dropout=0.0).to(device)
denoiser_2 = vit().to(device).eval()
ckpt = torch.load(cfg['ckpt_dir'], map_location='cpu')
denoiser_2.load_state_dict(ckpt['model'], strict=True)
denoiser_2.eval()

print(f'----------Loaded Denoiser--------------')

In [ ]:
state = load_file(cfg['ckpt_safetensors'])
model.load_state_dict(state, strict=True)
print(f'Loaded model')

In [ ]:
class InpaintOperator:
  def __init__(self, mask_ratio=.5, device='cpu'):
    self.mask_ratio = mask_ratio
    self.device = device
    self.mask = None

  def make_mask(self, shape):
    B, C, L, W = shape
    m = (torch.rand(B, C, L, W, device=self.device) > self.mask_ratio).float()
    self.mask = m
    return m

  def A(self, x):
    if self.mask is None:
      self.make_mask(x.shape)
    return self.mask * x

  def AT(self, y):
    print(f'A: {self.A(y).shape}')
    return self.A(y)

  def prox_f(self, y_obs, v, rho, sigma_n):
    """
    x = ((M/x^2)*y + rho*v) / ((M/s^2) + rho)
    """

    M = self.mask
    denominator = (M/(sigma_n**2)) + rho
    numerator = (M*y_obs / (sigma_n**2) + rho*v)

    """m_0 = sigma_n**2

    denominator = M + (m_0 * rho)
    numerator = (M * y_obs) + (m_0 * rho * v)"""

    return numerator / denominator #  1e-12

img_hu = np.random.normal(loc=-600, scale=300, size=(cfg['data']['img_size'], cfg['data']['img_size'])).astype(np.float32)
x_0 = hu_win_normalize(img_hu, cfg['data']['hu_min'], cfg['data']['hu_max'])
x_0 = torch.from_numpy(x_0)[None, None].to(next(model.parameters()).device) # [1, 1, L, W]

operator = InpaintOperator(mask_ratio=cfg['operator']['mask_ratio'], device=next(model.parameters()).device)
y_0 = operator.A(x_0)
y = y_0 + cfg['operator']['noise_sigma'] * torch.randn_like(y_0)


In [ ]:
def pnp_admm(y, op, denoiser, sigma_min, sigma_max, sigma_n, t_schedule_fn, x_init=None):
  # Move the input 'y' to the same device as the denoiser and operator
  y = y.to(next(denoiser.parameters()).device)

  iters = cfg['pnp']['iters']
  rho = cfg['pnp']['rho0']
  gamma = cfg['pnp']['gamma']
  lam = cfg['pnp']['lam']

  x = op.AT(y) if x_init is None else x_init
  z = x.clone()
  u = torch.zeros_like(x)

  logs = {
      'x': [],
      'z' : [],
      'u' : [],
      'rho': [],
      't': [],
      'primal' : [],
      'fp': [],
  }

  for k in range(iters):
    x_prev = x

    # x-update (closed form proxy for masking)
    x = op.prox_f(y_obs=y, v=z-u, rho=rho, sigma_n=sigma_n)

    # z-update (denoise)
    sigma_k = math.sqrt(lam/rho)
    sigma = torch.full((x.shape[0], 1, 1, 1), sigma_k, device=next(denoiser.parameters()).device)
    # Convert sigma (torch tensor) to numpy for t_mapping_log, then back to torch on device
    sigma_a = t_mapping_log_continuous(sigma, sigma_min, sigma_max)
    # sigma_a = torch.from_numpy(sigma_a).float().to(next(denoiser.parameters()).device)

    primal = (x - z).norm().item()

    t_k = t_schedule_fn(k, rho, primal, cfg)

    t_tensor = torch.full((x.shape[0], 1, 1, 1), t_k, device=next(denoiser.parameters()).device)
    z = denoiser(x + u, t_tensor) # Changed sigma_a to t_tensor here as per the common pattern for denoisers taking 't' as continuous time.
    

    # dual update

    u = u + (x - z)
    
    fp = ((x - x_prev).norm() /  (x_prev.norm() + 1e-12)).item()

    logs['x'].append(x.detach().cpu().numpy())
    logs['z'].append(z.detach().cpu().numpy())
    logs['u'].append(u.detach().cpu().numpy())
    
    logs['rho'].append(rho)
    logs['t'].append(t_k)
    logs['primal'].append(primal)
    logs['fp'].append(fp)
    logs['sigma'].append(sigma_k)

    if (primal < cfg['pnp']['total_primal']) and (fp < cfg['pnp']['total_fp']):
      print(f'Coverged at iter={k} | primal = {primal:.3e}, fp={fp:.3e}')
      break

    rho *= gamma

    np.savez(cfg['out_npz'],
         x = np.stack(logs['x']), z = np.stack(logs['z']), u = np.stack(logs['u']),
         rho=np.array(logs['rho']), t = np.array(logs['t']), primal = np.array(logs['primal']), fp = np.array(logs['fp']), sigma=np.array(logs['sigma']))
    
    return x, logs




In [ ]:
def sanity_check_module(model, rho=.1, sigma_n=.01):
  m = model(backbone, sigma_data=cfg['data']['sigma_data'])

  x_true = m

  v = torch.full_like(x_true, 0.5)

  op = InpaintOperator(mask_ratio=0.5, device=device)
  y_obs = op.A(x_true)

  x_next = op.prox_f(y_obs, v, rho, sigma_n)

  return x_true, op, y_obs, x_next

In [ ]:
x_true, op, y_obs, x_next = sanify_check_module(EMPPre)

In [ ]:
x_star, logs = pnp_admm(y=y, op=operator, denoiser=model, sigma_min=cfg['noise']['sigma_min'], sigma_max=cfg['noise']['sigma_max'], sigma=.01, t_schedule_fn=t_fixed)
x_star_adapt, logs_adapt = pnp_admm(y=y, op=operator, denoiser=model, sigma_min=cfg['noise']['sigma_min'], sigma_max=cfg['noise']['sigma_max'], sigma_n=.01, t_schedule_fn=t_rho_based)
print(f'length of pnp admm logs : {len(logs['primal'])} | {logs['primal'][-1]} | {logs['fp'][-1]}')

In [ ]:
# expected ∥x−y∥≈∥x−v∥≈1/2​∥y−v∥
for sigma_test in [1.0, 0.1, 0.01, 0.001]:

  x_test = op.prox_f(y_obs, v, rho, sigma_test)

  observed = op.mask == 1
  missing = op.mask == 0

  print("missing mean |y-v|      :", torch.norm((y_obs - v)[missing]).item())
  print("missing mean |x-y|      :", torch.norm((x_next - y_obs)[missing]).item())
  print("missing mean |x-v|      :", torch.norm((x_next - v)[missing]).item())

  d_yv = torch.norm((y_obs - v)[observed]).item()
  d_xy = torch.norm((x_test - y_obs)[observed]).item()
  d_xv = torch.norm((x_test - v)[observed]).item()

  print("||y-v||:", d_yv)
  print("||x-y||:", d_xy)
  print("||x-v||:", d_xv)
  print("ratio y:", d_xy / d_yv)
  print("ratio v:", d_xv / d_yv)

In [ ]:
out_dir = os.path.join(cfg['out_dir'], cfg['exp_id'])
os.makedirs(out_dir, exist_ok = True)

In [ ]:
# visualization
def show(img, title):
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(6, 4))
plt.plot(logs['primal'])
plt.yscale('log')
plt.title('Primal residual ||x - z||')
plt.xlabel('iter')
plt.ylabel('residual')
plt.savefig(os.path.join(out_dir, 'primal.png'), dpi = 72)

plt.figure(figsize=(6, 4))
plt.plot(logs['fp'])
plt.yscale('log')
plt.title('Fixed point ||x_k+1 - x_k|| / ||x_k||')
plt.xlabel('iter')
plt.ylabel('residual')
plt.savefig(os.path.join(out_dir, 'fixed_point.png'), dpi = 72)

plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1)
show(x_0[0, 0].detach().cpu(), 'x_0')
plt.subplot(1, 3, 2)
show(y[0, 0].detach().cpu(), 'y')
plt.subplot(1, 3, 3)
show(x_star[0, 0].detach().cpu(), 'x_start')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'reconstruction.png'), dpi = 72)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
show(x_star[0, 0].detach().cpu(), 'x_true (Ground Truth)')
plt.subplot(1, 3, 2)
show(y_obs[0, 0].detach().cpu(), 'y_obs (Observed/Masked)')
plt.subplot(1, 3, 3)
show(torch.abs(x_star - y_obs)[0, 0].detach().cpu(), '|x_true - y_obs| (Difference)')
plt.tight_layout()
plt.show()

In [ ]:
def save_logs(name, logs):
  path = os.path.join(out_dir, f'logs_{name}.csv')
  with open(path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['iter', 'rho', 't', 'primal', 'fp'])
    for i, (r, t, p, fp) in enumerate(zip(logs['rho'], logs['t'], logs['primal'], logs['fp'])):
      w.writerow([i,r,t,p,fp])

  print(f'saved: {out_dir}')

  return path


In [ ]:
p_1 = save_logs('fixed', logs)
p_2 = save_logs('adapt', logs_adapt)

torch.save(x_star.detach().cpu(), os.path.join(out_dir, 'x_star_fixed.pt'))
torch.save(x_star.detach().cpu(), os.path.join(out_dir, 'x_star_adapt.pt'))

print(f'saved: {out_dir}')
print(p_1); print(p_2)

In [ ]:
log_path = os.path.join(cfg['out_dir'], cfg['exp_id'], 'logs.csv')

df_logs = pd.read_csv(log_path)

display(df_logs.head())